# Laguna XS.2 — Final Causal Expert Surgery
## 1× RTX PRO 6000 Blackwell 96GB · 32 vCPU · 256GB RAM

This is the **direct-BF16 PyTorch/Transformers research path**.

No vLLM. No INT4 conversion. No tensor parallelism. No CPU model offload.

The notebook loads the official `poolside/Laguna-XS.2` BF16 checkpoint and performs:

1. hardware/storage/memory preflight;
2. direct BF16 model loading;
3. exact fixed-routing causal layer ablation;
4. hierarchical expert-group search;
5. exact individual expert validation;
6. bootstrap confidence intervals;
7. renormalization robustness;
8. routing-frequency vs causal-rank comparison;
9. optional expert-pair interactions;
10. optional **full-rank training of only selected expert slices** through a compact external trainable expert bank.

### Hardware target

- GPU: RTX PRO 6000 Blackwell 96GB (~89 GiB binary)
- CPU: 32 vCPU
- system RAM: 256GB
- local SSD/NVMe: ideally 100GB+ free

### Model facts validated in-notebook

Expected Laguna XS.2 configuration:

- 40 transformer layers
- 39 sparse MoE layers
- hidden size 2048
- 256 routed experts per sparse layer
- top-8 routing
- expert width 512
- fused expert tensors:
  - `gate_up_proj[256, 1024, 2048]`
  - `down_proj[256, 2048, 512]`

### Core causal score

\[
S(E)=\Delta L_{target}(E)-\lambda\max(0,\Delta L_{control}(E))
\]

Routing frequency is measured only **after** causal selection.

> **Final hardening:** explicitly registers Transformers' native Laguna
> checkpoint-conversion mapping before BF16 loading and validates all critical
> MoE keys before causal experiments begin.


## 1 — Install research dependencies

In [ ]:
# Keep the environment's CUDA-enabled PyTorch. Do not reinstall torch.
%pip -q install -U \
  "transformers==5.14.1" \
  "accelerate>=1.10.0" \
  "huggingface_hub>=0.35.0" \
  safetensors pandas numpy psutil tqdm matplotlib scikit-learn

## 2 — Runtime tuning for 32 vCPU / 256GB RAM

In [ ]:
import os

os.environ["OMP_NUM_THREADS"] = "16"
os.environ["MKL_NUM_THREADS"] = "16"
os.environ["OPENBLAS_NUM_THREADS"] = "16"
os.environ["NUMEXPR_NUM_THREADS"] = "16"
os.environ["TOKENIZERS_PARALLELISM"] = "true"
os.environ["MALLOC_ARENA_MAX"] = "8"

os.environ["HF_ENABLE_PARALLEL_LOADING"] = "true"
os.environ["HF_PARALLEL_LOADING_WORKERS"] = "8"
os.environ["HF_XET_NUM_CONCURRENT_RANGE_GETS"] = "16"
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"

os.environ["CUDA_MODULE_LOADING"] = "LAZY"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:512"

print("Runtime environment configured.")

## 3 — Hardware and storage preflight

In [ ]:
import os, shutil, platform
from pathlib import Path
import psutil
import torch

ram = psutil.virtual_memory()

print("=== Host ===")
print("Python:", platform.python_version())
print("Logical CPUs:", os.cpu_count())
print(f"RAM total:     {ram.total/2**30:.2f} GiB")
print(f"RAM available: {ram.available/2**30:.2f} GiB")

print("\n=== CUDA ===")
print("Torch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not visible.")

if torch.cuda.device_count() != 1:
    raise RuntimeError(f"Expected exactly one GPU; found {torch.cuda.device_count()}.")

props = torch.cuda.get_device_properties(0)
print("GPU:", props.name)
print(f"VRAM: {props.total_memory/2**30:.2f} GiB")
print("Compute capability:", torch.cuda.get_device_capability(0))

if props.total_memory / 2**30 < 88:
    raise RuntimeError("Need a 96GB-class GPU (~89 GiB binary).")

if (os.cpu_count() or 0) < 24:
    print("WARNING: fewer than 24 logical CPUs detected.")

if ram.total / 2**30 < 220:
    print("WARNING: less than ~256GB-class RAM detected.")

roots = [
    Path("/home/ec2-user/workspace"),
    Path("/workspace"),
    Path("/mnt/data"),
    Path("/root"),
    Path("/tmp"),
    Path.cwd(),
]

choices = []
seen = set()
for p in roots:
    try:
        if not p.exists() or not os.access(p, os.W_OK):
            continue
        dev = os.stat(p).st_dev
        if dev in seen:
            continue
        seen.add(dev)
        usage = shutil.disk_usage(p)
        choices.append((usage.free, p, usage))
    except OSError:
        pass

if not choices:
    raise RuntimeError("No writable filesystem found.")

_, WORK_ROOT, disk = max(choices, key=lambda x: x[0])

print("\n=== Storage ===")
print("Work root:", WORK_ROOT)
print(f"Disk free: {disk.free/2**30:.2f} GiB")
print("Hardware preflight: PASS")

## 4 — Resolve the official BF16 checkpoint

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

MODEL_ID = "poolside/Laguna-XS.2"

default_model_path = (
    Path("/home/ec2-user/workspace/models/Laguna-XS.2")
    if Path("/home/ec2-user/workspace").exists()
    else WORK_ROOT / "models" / "Laguna-XS.2"
)

MODEL_PATH = Path(
    os.environ.get("LAGUNA_BF16_PATH", str(default_model_path))
).expanduser().resolve()

EXPECTED_SHARDS = [
    f"model-{i:05d}-of-00014.safetensors"
    for i in range(1, 15)
]

complete = (
    (MODEL_PATH / "config.json").exists()
    and all((MODEL_PATH / x).exists() for x in EXPECTED_SHARDS)
)

if not complete:
    MODEL_PATH.mkdir(parents=True, exist_ok=True)
    free_gib = shutil.disk_usage(MODEL_PATH).free / 2**30

    if free_gib < 85:
        raise RuntimeError(
            f"Need ~85 GiB free for a fresh BF16 download; found {free_gib:.1f} GiB. "
            "Mount the checkpoint elsewhere and set LAGUNA_BF16_PATH."
        )

    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=str(MODEL_PATH),
        allow_patterns=[
            "*.safetensors", "*.json", "*.py", "*.jinja",
            "LICENSE*", "README*",
        ],
        max_workers=8,
    )

missing = [x for x in EXPECTED_SHARDS if not (MODEL_PATH / x).exists()]
if missing:
    raise RuntimeError(f"Incomplete BF16 checkpoint; missing: {missing}")

sizes = [(x, (MODEL_PATH / x).stat().st_size) for x in EXPECTED_SHARDS]

print("MODEL_PATH:", MODEL_PATH)
print(f"14-shard tensor payload: {sum(n for _, n in sizes)/1e9:.3f} GB")
print(f"First shard: {sizes[0][1]/1e9:.3f} GB")
print(f"Last shard:  {sizes[-1][1]/1e9:.3f} GB")
print("Checkpoint verification: PASS")

## 5 — Explicit Laguna checkpoint-conversion registration

Transformers' dynamic weight loader can consolidate per-expert checkpoint
tensors into Laguna's fused 3D expert tensors. For `trust_remote_code` models,
native mappings may be skipped unless explicitly registered.

Register the built-in `laguna` mapping before the 66.9 GB BF16 load. This is
idempotent and prevents the exact class of per-expert/fused-key mismatch seen
with the quantized checkpoint.

In [ ]:
from transformers.conversion_mapping import (
    get_checkpoint_conversion_mapping,
    register_checkpoint_conversion_mapping,
    USER_REGISTERED_MAPPINGS,
)

laguna_mapping = get_checkpoint_conversion_mapping("laguna")

if laguna_mapping is None:
    raise RuntimeError(
        "This Transformers installation does not expose the native Laguna "
        "checkpoint conversion mapping."
    )

register_checkpoint_conversion_mapping(
    "laguna",
    laguna_mapping,
    overwrite=True,
)

if "laguna" not in USER_REGISTERED_MAPPINGS:
    raise RuntimeError("Laguna conversion mapping registration failed.")

print("Laguna checkpoint conversion mapping: REGISTERED")
print("Conversion operations:", len(laguna_mapping))

## 6 — Load BF16 directly onto the RTX PRO 6000

In [ ]:
import gc, time, torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.set_num_threads(min(16, os.cpu_count() or 16))
try:
    torch.set_num_interop_threads(4)
except RuntimeError:
    pass

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

print("Transformers:", transformers.__version__)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    fix_mistral_regex=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

t0 = time.time()

model, loading_info = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    device_map={"": 0},
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    use_safetensors=True,
    attn_implementation="sdpa",
    output_loading_info=True,
)

model.eval()
model.config.use_cache = False

for p in model.parameters():
    p.requires_grad_(False)

missing = list(loading_info.get("missing_keys", []))
unexpected = list(loading_info.get("unexpected_keys", []))
mismatched = list(loading_info.get("mismatched_keys", []))

critical = [
    k for k in (missing + unexpected)
    if ".mlp.experts" in k or ".mlp.gate" in k or "e_score_correction_bias" in k
]

if critical or mismatched:
    raise RuntimeError(
        "Critical checkpoint mismatch.\n"
        f"critical sample: {critical[:12]}\n"
        f"mismatched sample: {mismatched[:12]}"
    )

del loading_info
gc.collect()
torch.cuda.synchronize()

free_b, total_b = torch.cuda.mem_get_info()

print(f"Loaded in {(time.time()-t0)/60:.2f} min")
print(f"GPU allocated: {torch.cuda.memory_allocated()/2**30:.2f} GiB")
print(f"GPU reserved:  {torch.cuda.memory_reserved()/2**30:.2f} GiB")
print(f"GPU peak:      {torch.cuda.max_memory_allocated()/2**30:.2f} GiB")
print(f"Driver free:   {free_b/2**30:.2f} GiB")
print(f"Host RAM available: {psutil.virtual_memory().available/2**30:.2f} GiB")

print("BF16 load: PASS")

## 6 — Validate exact Laguna MoE structure

In [ ]:
cfg = model.config

SPARSE_LAYERS = []
for idx, layer in enumerate(model.model.layers):
    mlp = getattr(layer, "mlp", None)
    if (
        mlp is not None
        and hasattr(mlp, "gate")
        and hasattr(mlp, "experts")
        and hasattr(mlp.experts, "gate_up_proj")
        and hasattr(mlp.experts, "down_proj")
    ):
        SPARSE_LAYERS.append(idx)

print("hidden_size:", cfg.hidden_size)
print("layers:", cfg.num_hidden_layers)
print("experts:", cfg.num_experts)
print("top_k:", cfg.num_experts_per_tok)
print("expert width:", cfg.moe_intermediate_size)
print("sparse layers:", SPARSE_LAYERS)

assert cfg.hidden_size == 2048
assert cfg.num_hidden_layers == 40
assert cfg.num_experts == 256
assert cfg.num_experts_per_tok == 8
assert cfg.moe_intermediate_size == 512
assert len(SPARSE_LAYERS) == 39

sample = model.model.layers[SPARSE_LAYERS[0]].mlp
print("gate_up_proj:", tuple(sample.experts.gate_up_proj.shape), sample.experts.gate_up_proj.dtype)
print("down_proj:   ", tuple(sample.experts.down_proj.shape), sample.experts.down_proj.dtype)
print("router:", tuple(sample.gate.weight.shape), sample.gate.weight.dtype)

assert tuple(sample.experts.gate_up_proj.shape) == (256, 1024, 2048)
assert tuple(sample.experts.down_proj.shape) == (256, 2048, 512)

params_per_expert = (
    sample.experts.gate_up_proj[0].numel()
    + sample.experts.down_proj[0].numel()
)

print(f"params/expert: {params_per_expert:,} ({params_per_expert/1e6:.3f}M)")
print("MoE structure: PASS")

## 7 — Short forward smoke test

In [ ]:
@torch.inference_mode()
def smoke_forward(text):
    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
    )
    enc = {k: v.to("cuda:0", non_blocking=True) for k, v in enc.items()}

    return model(
        **enc,
        use_cache=False,
        logits_to_keep=1,
        return_dict=True,
    ).logits

torch.cuda.reset_peak_memory_stats()

logits = smoke_forward(
    "Explain briefly why min-width: 0 can matter inside a CSS flex container."
)

torch.cuda.synchronize()
free_b, _ = torch.cuda.mem_get_info()

print("logits shape:", tuple(logits.shape))
print(f"Peak allocation: {torch.cuda.max_memory_allocated()/2**30:.2f} GiB")
print(f"Driver free: {free_b/2**30:.2f} GiB")

del logits
torch.cuda.empty_cache()

# Phase A — Target/control evaluation set

The bundled examples are only a pipeline validation set.

For publication-quality work, replace them with:
- 50–200+ target-selection examples,
- 50–200+ matched controls,
- a separate held-out evaluation set.

In [ ]:
import pandas as pd

EVAL_ROWS = [
    {"kind":"target","prompt":"In CSS flexbox, what declaration lets a flex item shrink below its intrinsic content width?","reference":"min-width: 0;"},
    {"kind":"target","prompt":"What CSS declaration establishes a flex formatting context?","reference":"display: flex;"},
    {"kind":"target","prompt":"Which React hook stores local component state?","reference":"useState"},
    {"kind":"target","prompt":"What CSS property controls horizontal overflow?","reference":"overflow-x"},
    {"kind":"target","prompt":"Which CSS property controls stacking order for positioned elements?","reference":"z-index"},
    {"kind":"target","prompt":"Which declaration commonly centers flex children along the main axis?","reference":"justify-content: center;"},
    {"kind":"target","prompt":"In React, which prop gives a stable identity to list items?","reference":"key"},
    {"kind":"target","prompt":"Which CSS property sets spacing between grid or flex children without margins?","reference":"gap"},
    {"kind":"target","prompt":"Which CSS property includes padding and border inside declared width?","reference":"box-sizing"},
    {"kind":"target","prompt":"Which browser API observes element size changes?","reference":"ResizeObserver"},
    {"kind":"target","prompt":"Which React hook runs side effects after rendering?","reference":"useEffect"},
    {"kind":"target","prompt":"Which CSS property defines grid columns?","reference":"grid-template-columns"},

    {"kind":"control","prompt":"Which Python keyword yields a value from a generator?","reference":"yield"},
    {"kind":"control","prompt":"Which traversal finds shortest paths in an unweighted graph?","reference":"BFS"},
    {"kind":"control","prompt":"Which Java keyword declares class inheritance?","reference":"extends"},
    {"kind":"control","prompt":"Which SQL keyword removes duplicate SELECT rows?","reference":"DISTINCT"},
    {"kind":"control","prompt":"Which C++ smart pointer represents exclusive ownership?","reference":"std::unique_ptr"},
    {"kind":"control","prompt":"Which asymptotic notation describes an upper bound?","reference":"Big O"},
    {"kind":"control","prompt":"Which Python container provides average O(1) membership lookup for hashable values?","reference":"set"},
    {"kind":"control","prompt":"Which data structure is first-in first-out?","reference":"queue"},
    {"kind":"control","prompt":"Which SQL clause filters groups after aggregation?","reference":"HAVING"},
    {"kind":"control","prompt":"Which Java interface defines natural ordering?","reference":"Comparable"},
    {"kind":"control","prompt":"Which C++ keyword prevents modification through that name?","reference":"const"},
    {"kind":"control","prompt":"What mathematical operation is the inverse of exponentiation for solving an exponent?","reference":"logarithm"},
]

eval_df = pd.DataFrame(EVAL_ROWS)
display(eval_df.groupby("kind").size().rename("count"))

## 8 — Build one reusable aligned GPU scoring batch

In [ ]:
def chat_prefix_text(prompt):
    messages = [{"role":"user","content":prompt}]
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

def parse_case(prompt, reference):
    prefix_text = chat_prefix_text(prompt)
    prefix_ids = tokenizer.encode(prefix_text, add_special_tokens=False)
    full_ids = tokenizer.encode(prefix_text + " " + reference, add_special_tokens=False)

    start = 0
    for a, b in zip(prefix_ids, full_ids):
        if a != b:
            break
        start += 1

    if start <= 0 or start >= len(full_ids):
        raise RuntimeError(f"Could not identify answer boundary for {prompt!r}")

    return full_ids, start

def build_scoring_batch(df):
    parsed = [parse_case(r.prompt, r.reference) for r in df.itertuples(index=False)]
    fulls = [x[0] for x in parsed]
    starts = [x[1] for x in parsed]
    answers = [full[start:] for full, start in zip(fulls, starts)]

    max_prefix = max(starts)
    max_ref = max(len(x) for x in answers)
    batch_size = len(df)
    seq_len = max_prefix + max_ref

    input_ids = torch.full(
        (batch_size, seq_len),
        tokenizer.pad_token_id,
        dtype=torch.long,
    )
    attention_mask = torch.zeros((batch_size, seq_len), dtype=torch.long)
    targets = torch.full((batch_size, max_ref), -100, dtype=torch.long)

    for b, (full, start, ans) in enumerate(zip(fulls, starts, answers)):
        prefix = full[:start]
        prefix_start = max_prefix - len(prefix)

        input_ids[b, prefix_start:max_prefix] = torch.tensor(prefix)
        attention_mask[b, prefix_start:max_prefix] = 1

        input_ids[b, max_prefix:max_prefix+len(ans)] = torch.tensor(ans)
        attention_mask[b, max_prefix:max_prefix+len(ans)] = 1
        targets[b, :len(ans)] = torch.tensor(ans)

    position_ids = attention_mask.cumsum(dim=-1) - 1
    position_ids.clamp_(min=0)

    pred_positions = torch.arange(
        max_prefix - 1,
        max_prefix + max_ref - 1,
        dtype=torch.long,
    )

    return {
        "input_ids": input_ids.to("cuda:0", non_blocking=True),
        "attention_mask": attention_mask.to("cuda:0", non_blocking=True),
        "position_ids": position_ids.to("cuda:0", non_blocking=True),
        "targets": targets.to("cuda:0", non_blocking=True),
        "pred_positions": pred_positions.to("cuda:0", non_blocking=True),
    }

SCORE_BATCH = build_scoring_batch(eval_df)

print("Batch:", SCORE_BATCH["input_ids"].shape)
print("Reference positions:", len(SCORE_BATCH["pred_positions"]))

## 9 — Cached teacher-forced reference NLL

In [ ]:
import numpy as np
import torch.nn.functional as F

@torch.inference_mode()
def score_cached_batch():
    out = model(
        input_ids=SCORE_BATCH["input_ids"],
        attention_mask=SCORE_BATCH["attention_mask"],
        position_ids=SCORE_BATCH["position_ids"],
        use_cache=False,
        output_router_logits=False,
        logits_to_keep=SCORE_BATCH["pred_positions"],
        return_dict=True,
    )

    logits = out.logits.float()
    targets = SCORE_BATCH["targets"]

    losses = F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]),
        targets.reshape(-1),
        ignore_index=-100,
        reduction="none",
    ).reshape(targets.shape)

    valid = targets.ne(-100)
    per_example = (losses * valid).sum(-1) / valid.sum(-1).clamp_min(1)
    result = per_example.detach().cpu().numpy()

    del out, logits, losses, per_example
    return result

BASE_NLL = score_cached_batch()

target_mask = eval_df["kind"].values == "target"
control_mask = eval_df["kind"].values == "control"

print("Baseline target NLL: ", float(BASE_NLL[target_mask].mean()))
print("Baseline control NLL:", float(BASE_NLL[control_mask].mean()))

# Phase B — Fixed-routing causal intervention

The router is called normally first.

The patch changes only returned `routing_weights`; `selected_experts` remains
unchanged, so no replacement expert is allowed to enter the top-k.

In [ ]:
from contextlib import contextmanager, ExitStack
import types

def get_sparse_mlp(layer_idx):
    layer_idx = int(layer_idx)
    if layer_idx not in SPARSE_LAYERS:
        raise ValueError(f"Layer {layer_idx} is not sparse.")
    return model.model.layers[layer_idx].mlp

@contextmanager
def gate_intervention(layer_idx, expert_ids=None, zero_all_routed=False, renormalize=False):
    mlp = get_sparse_mlp(layer_idx)
    gate = mlp.gate
    original_forward = gate.forward
    expert_ids = [] if expert_ids is None else [int(x) for x in expert_ids]

    def patched_forward(self, hidden_states):
        router_logits, routing_weights, selected_experts = original_forward(hidden_states)

        if zero_all_routed:
            routing_weights = torch.zeros_like(routing_weights)

        elif expert_ids:
            ids = torch.tensor(
                expert_ids,
                device=selected_experts.device,
                dtype=selected_experts.dtype,
            )
            keep = ~torch.isin(selected_experts, ids)
            routing_weights = routing_weights * keep.to(routing_weights.dtype)

            if renormalize:
                denom = routing_weights.sum(-1, keepdim=True)
                routing_weights = torch.where(
                    denom > 0,
                    routing_weights / denom.clamp_min(1e-12),
                    routing_weights,
                )

        return router_logits, routing_weights, selected_experts

    gate.forward = types.MethodType(patched_forward, gate)
    try:
        yield
    finally:
        gate.forward = original_forward

## 10 — Intervention integrity check

In [ ]:
test_layer = SPARSE_LAYERS[len(SPARSE_LAYERS)//2]

with gate_intervention(test_layer, zero_all_routed=True):
    test_nll = score_cached_batch()

delta = test_nll - BASE_NLL

print("Test layer:", test_layer)
print("Mean |ΔNLL|:", float(np.abs(delta).mean()))

if np.abs(delta).mean() < 1e-7:
    raise RuntimeError("Intervention produced no meaningful change.")

print("Fixed-routing intervention: PASS")

## 11 — Causal layer sweep

In [ ]:
from tqdm.auto import tqdm
import time

CONTROL_PENALTY = 0.75

def summarize_delta(ablated):
    delta = np.asarray(ablated) - BASE_NLL
    td = float(delta[target_mask].mean())
    cd = float(delta[control_mask].mean())

    return {
        "target_delta_nll": td,
        "control_delta_nll": cd,
        "causal_specificity": td - CONTROL_PENALTY * max(cd, 0.0),
        "per_example_delta": delta,
    }

RESULTS = WORK_ROOT / "laguna_xs2_causal_surgery_results"
RESULTS.mkdir(parents=True, exist_ok=True)

layer_rows = []
t0 = time.time()

for layer_idx in tqdm(SPARSE_LAYERS, desc="39-layer causal sweep"):
    with gate_intervention(layer_idx, zero_all_routed=True):
        nll = score_cached_batch()

    m = summarize_delta(nll)
    layer_rows.append({
        "layer": int(layer_idx),
        "target_delta_nll": m["target_delta_nll"],
        "control_delta_nll": m["control_delta_nll"],
        "causal_specificity": m["causal_specificity"],
    })

layer_df = pd.DataFrame(layer_rows).sort_values(
    "causal_specificity",
    ascending=False,
).reset_index(drop=True)

layer_df.to_csv(RESULTS / "layer_causal_scores.csv", index=False)

print(f"Layer sweep: {(time.time()-t0)/60:.2f} min")
display(layer_df.head(15))

## 12 — Hierarchical expert-group search

In [ ]:
def intervention_score(layer_idx, expert_ids, renormalize=False):
    with gate_intervention(
        layer_idx,
        expert_ids=expert_ids,
        renormalize=renormalize,
    ):
        nll = score_cached_batch()

    return summarize_delta(nll)

def hierarchical_expert_search(
    layer_idx,
    seed=17,
    initial_group_size=32,
    beam_width=3,
):
    rng = np.random.default_rng(seed)
    order = rng.permutation(cfg.num_experts).tolist()

    frontier = [
        order[i:i+initial_group_size]
        for i in range(0, len(order), initial_group_size)
    ]

    history = []
    level = 0

    while frontier:
        current = []

        for group in tqdm(
            frontier,
            desc=f"L{layer_idx} level {level}",
            leave=False,
        ):
            m = intervention_score(layer_idx, group)

            rec = {
                "layer": int(layer_idx),
                "seed": int(seed),
                "level": int(level),
                "group_size": len(group),
                "experts": list(map(int, group)),
                "target_delta_nll": m["target_delta_nll"],
                "control_delta_nll": m["control_delta_nll"],
                "causal_specificity": m["causal_specificity"],
            }

            history.append(rec)
            current.append(rec)

        current.sort(key=lambda x: x["causal_specificity"], reverse=True)
        keep = current[:beam_width]

        if all(x["group_size"] == 1 for x in keep):
            break

        nxt = []
        for rec in keep:
            g = rec["experts"]
            if len(g) == 1:
                nxt.append(g)
            else:
                mid = len(g)//2
                nxt.extend([g[:mid], g[mid:]])

        frontier = [x for x in nxt if x]
        level += 1

    hist = pd.DataFrame(history)
    leaf_size = hist["group_size"].min()
    leaves = hist[hist.group_size == leaf_size].sort_values(
        "causal_specificity",
        ascending=False,
    )

    return hist, leaves

## 13 — Search top causal layers with two expert-order seeds

In [ ]:
TOP_LAYERS = 4
SEARCH_SEEDS = [17, 53]
INITIAL_GROUP_SIZE = 32
BEAM_WIDTH = 3

candidate_layers = layer_df.head(TOP_LAYERS)["layer"].astype(int).tolist()
print("Candidate layers:", candidate_layers)

histories = []
leaf_frames = []

for layer_idx in candidate_layers:
    for seed in SEARCH_SEEDS:
        hist, leaves = hierarchical_expert_search(
            layer_idx,
            seed=seed,
            initial_group_size=INITIAL_GROUP_SIZE,
            beam_width=BEAM_WIDTH,
        )
        histories.append(hist)
        leaf_frames.append(leaves)

group_history = pd.concat(histories, ignore_index=True)
leaf_df = pd.concat(leaf_frames, ignore_index=True)

group_history.to_json(
    RESULTS / "hierarchical_group_history.json",
    orient="records",
    indent=2,
)

display(leaf_df.head(30))

## 14 — Exact individual expert validation

In [ ]:
leaf_pairs = sorted({
    (int(r.layer), int(e))
    for r in leaf_df.itertuples(index=False)
    for e in r.experts
})

print("Unique leaf candidates:", len(leaf_pairs))

individual_rows = []

for layer_idx, expert_id in tqdm(leaf_pairs, desc="Individual validation"):
    m = intervention_score(layer_idx, [expert_id])

    individual_rows.append({
        "layer": layer_idx,
        "expert": expert_id,
        "target_delta_nll": m["target_delta_nll"],
        "control_delta_nll": m["control_delta_nll"],
        "causal_specificity": m["causal_specificity"],
        "per_example_delta": m["per_example_delta"].tolist(),
    })

individual_df = pd.DataFrame(individual_rows).sort_values(
    "causal_specificity",
    ascending=False,
).reset_index(drop=True)

individual_df.drop(columns=["per_example_delta"]).to_csv(
    RESULTS / "individual_causal_experts.csv",
    index=False,
)

display(individual_df.head(25))

## 15 — Bootstrap confidence intervals

In [ ]:
def bootstrap_specificity(delta, n_boot=5000, seed=123):
    rng = np.random.default_rng(seed)
    d = np.asarray(delta, dtype=np.float64)

    t = d[target_mask]
    c = d[control_mask]

    vals = np.empty(n_boot, dtype=np.float64)

    for i in range(n_boot):
        tb = rng.choice(t, size=len(t), replace=True).mean()
        cb = rng.choice(c, size=len(c), replace=True).mean()
        vals[i] = tb - CONTROL_PENALTY * max(cb, 0.0)

    return {
        "ci_2.5": float(np.quantile(vals, 0.025)),
        "ci_97.5": float(np.quantile(vals, 0.975)),
        "p_positive": float((vals > 0).mean()),
    }

boot_rows = []

for r in individual_df.itertuples(index=False):
    boot_rows.append({
        "layer": int(r.layer),
        "expert": int(r.expert),
        **bootstrap_specificity(r.per_example_delta),
    })

boot_df = pd.DataFrame(boot_rows)

final_df = individual_df.merge(
    boot_df,
    on=["layer","expert"],
    how="left",
).sort_values(
    ["p_positive","causal_specificity"],
    ascending=False,
).reset_index(drop=True)

display(final_df.head(25))

## 16 — Renormalization robustness

In [ ]:
ROBUST_TOP_N = min(16, len(final_df))

robust_rows = []

for r in tqdm(
    list(final_df.head(ROBUST_TOP_N).itertuples(index=False)),
    desc="Renormalized validation",
):
    m = intervention_score(
        int(r.layer),
        [int(r.expert)],
        renormalize=True,
    )

    robust_rows.append({
        "layer": int(r.layer),
        "expert": int(r.expert),
        "renorm_target_delta_nll": m["target_delta_nll"],
        "renorm_control_delta_nll": m["control_delta_nll"],
        "renorm_causal_specificity": m["causal_specificity"],
    })

robust_df = pd.DataFrame(robust_rows)

final_robust = final_df.merge(
    robust_df,
    on=["layer","expert"],
    how="left",
)

final_robust.drop(columns=["per_example_delta"]).to_csv(
    RESULTS / "final_candidates_with_renorm.csv",
    index=False,
)

display(final_robust.head(20))

## 17 — Routing diagnostic after causal selection

In [ ]:
@contextmanager
def capture_routing():
    records = {}
    originals = []
    valid_flat = SCORE_BATCH["attention_mask"].reshape(-1).bool()

    for layer_idx in SPARSE_LAYERS:
        gate = get_sparse_mlp(layer_idx).gate
        original = gate.forward
        originals.append((gate, original))

        def make_forward(idx, original_forward):
            def patched(self, hidden_states):
                logits, weights, selected = original_forward(hidden_states)

                with torch.no_grad():
                    mask = valid_flat
                    if mask.numel() == selected.shape[0]:
                        mask = mask.to(selected.device)
                    else:
                        mask = torch.ones(
                            selected.shape[0],
                            device=selected.device,
                            dtype=torch.bool,
                        )

                    ids = selected[mask].reshape(-1).long()
                    ws = weights[mask].reshape(-1).float()

                    counts = torch.bincount(ids, minlength=cfg.num_experts)
                    wsum = torch.zeros(
                        cfg.num_experts,
                        device=ws.device,
                        dtype=torch.float32,
                    )
                    wsum.scatter_add_(0, ids, ws)

                    records[int(idx)] = {
                        "tokens": int(mask.sum().item()),
                        "counts": counts.cpu(),
                        "weight_sums": wsum.cpu(),
                    }

                return logits, weights, selected
            return patched

        gate.forward = types.MethodType(
            make_forward(layer_idx, original),
            gate,
        )

    try:
        yield records
    finally:
        for gate, original in originals:
            gate.forward = original

with capture_routing() as routing_records:
    _ = score_cached_batch()

routing_rows = []

for layer_idx, rec in routing_records.items():
    tokens = max(1, rec["tokens"])

    for expert_id in range(cfg.num_experts):
        count = int(rec["counts"][expert_id].item())
        if count == 0:
            continue

        routing_rows.append({
            "layer": layer_idx,
            "expert": expert_id,
            "selected_rate": count / tokens,
            "routing_mass": float(rec["weight_sums"][expert_id].item()) / tokens,
        })

routing_df = pd.DataFrame(routing_rows)

comparison = final_robust.merge(
    routing_df,
    on=["layer","expert"],
    how="left",
).fillna({"selected_rate":0.0, "routing_mass":0.0})

comparison["causal_rank"] = comparison["causal_specificity"].rank(
    ascending=False, method="min"
)
comparison["routing_rank"] = comparison["routing_mass"].rank(
    ascending=False, method="min"
)
comparison["rank_gap"] = comparison["routing_rank"] - comparison["causal_rank"]

comparison.drop(columns=["per_example_delta"]).to_csv(
    RESULTS / "routing_vs_causality.csv",
    index=False,
)

display(
    comparison[
        [
            "layer","expert","causal_specificity","routing_mass",
            "causal_rank","routing_rank","rank_gap",
        ]
    ].sort_values("causal_rank").head(25)
)

## 18 — Optional same-layer expert-pair interactions

In [ ]:
from itertools import combinations

RUN_PAIR_TESTS = True
PAIR_TOP_N = min(10, len(final_robust))

pair_rows = []

if RUN_PAIR_TESTS:
    top = final_robust.head(PAIR_TOP_N)

    for layer_idx, group in top.groupby("layer"):
        rows = list(group.itertuples(index=False))

        for a, b in combinations(rows, 2):
            m = intervention_score(
                int(layer_idx),
                [int(a.expert), int(b.expert)],
            )

            pair_rows.append({
                "layer": int(layer_idx),
                "expert_a": int(a.expert),
                "expert_b": int(b.expert),
                "pair_causal_specificity": m["causal_specificity"],
                "interaction_score": (
                    m["causal_specificity"]
                    - float(a.causal_specificity)
                    - float(b.causal_specificity)
                ),
            })

pair_df = pd.DataFrame(pair_rows)

if not pair_df.empty:
    pair_df = pair_df.sort_values(
        "pair_causal_specificity",
        ascending=False,
    )
    pair_df.to_csv(
        RESULTS / "same_layer_expert_interactions.csv",
        index=False,
    )
    display(pair_df)
else:
    print("No same-layer top-candidate pairs.")

# Phase F — Selected-expert full-rank surgery

The full fused expert tensors remain frozen.

Only validated `(layer, expert)` slices are copied into a small external
trainable parameter bank.

The expert forward pass is monkeypatched only in layers that contain selected
experts:

- unselected expert → original frozen BF16 slice;
- selected expert → FP32 trainable master from the bank, autocast to BF16 for GEMM.

This avoids optimizer states for all 256 experts.

In [ ]:
import torch.nn as nn

AUTO_SELECTED = (
    final_robust[
        (final_robust["p_positive"] >= 0.95)
        & (final_robust["causal_specificity"] > 0)
        & (
            final_robust["renorm_causal_specificity"].isna()
            | (final_robust["renorm_causal_specificity"] > 0)
        )
    ]
    .head(8)[["layer","expert"]]
)

SELECTED_EXPERTS = [
    (int(r.layer), int(r.expert))
    for r in AUTO_SELECTED.itertuples(index=False)
]

print("Automatic selected experts:", SELECTED_EXPERTS)

# Optional manual override:
# SELECTED_EXPERTS = [(18, 93), (22, 41)]

In [ ]:
class SurgicalExpertBank(nn.Module):
    # Trainable FP32 copies of selected Laguna expert slices only.

    def __init__(self, selected_pairs):
        super().__init__()

        self.selected_pairs = sorted({
            (int(l), int(e))
            for l, e in selected_pairs
        })

        self.params = nn.ParameterDict()
        self.key_map = {}
        self.original_forwards = {}
        self.installed = False

        for layer_idx, expert_id in self.selected_pairs:
            experts = get_sparse_mlp(layer_idx).experts

            gu_key = f"L{layer_idx}_E{expert_id}_gu"
            down_key = f"L{layer_idx}_E{expert_id}_down"

            self.params[gu_key] = nn.Parameter(
                experts.gate_up_proj[expert_id].detach().float().clone()
            )
            self.params[down_key] = nn.Parameter(
                experts.down_proj[expert_id].detach().float().clone()
            )

            self.key_map[(layer_idx, expert_id)] = (gu_key, down_key)

        self.to("cuda:0")

    @property
    def trainable_parameter_count(self):
        return sum(p.numel() for p in self.parameters())

    def install(self):
        if self.installed:
            return

        for layer_idx in sorted({l for l, _ in self.selected_pairs}):
            experts = get_sparse_mlp(layer_idx).experts
            original = experts.forward
            self.original_forwards[layer_idx] = original

            selected_ids = {
                e for l, e in self.selected_pairs if l == layer_idx
            }
            bank = self

            def make_forward(idx, base_experts, selected):
                def surgical_forward(
                    self_experts,
                    hidden_states,
                    top_k_index,
                    top_k_weights,
                ):
                    final_hidden_states = torch.zeros_like(hidden_states)

                    with torch.no_grad():
                        expert_mask = F.one_hot(
                            top_k_index,
                            num_classes=self_experts.num_experts,
                        ).permute(2, 1, 0)

                        expert_hit = torch.greater(
                            expert_mask.sum(dim=(-1, -2)),
                            0,
                        ).nonzero(as_tuple=False).reshape(-1)

                    for expert_tensor in expert_hit:
                        expert_id = int(expert_tensor.item())
                        top_k_pos, token_idx = torch.where(
                            expert_mask[expert_id]
                        )
                        current_state = hidden_states[token_idx]

                        if expert_id in selected:
                            gu_key, down_key = bank.key_map[(idx, expert_id)]
                            gu = bank.params[gu_key].to(current_state.dtype)
                            down = bank.params[down_key].to(current_state.dtype)
                        else:
                            gu = base_experts.gate_up_proj[expert_id]
                            down = base_experts.down_proj[expert_id]

                        gate, up = F.linear(
                            current_state,
                            gu,
                        ).chunk(2, dim=-1)

                        h = base_experts.act_fn(gate) * up
                        h = F.linear(h, down)
                        h = h * top_k_weights[token_idx, top_k_pos, None]

                        final_hidden_states.index_add_(
                            0,
                            token_idx,
                            h.to(final_hidden_states.dtype),
                        )

                    return final_hidden_states

                return surgical_forward

            experts.forward = types.MethodType(
                make_forward(layer_idx, experts, selected_ids),
                experts,
            )

        self.installed = True

    def restore(self):
        if not self.installed:
            return

        for layer_idx, original in self.original_forwards.items():
            get_sparse_mlp(layer_idx).experts.forward = original

        self.original_forwards.clear()
        self.installed = False

    @torch.no_grad()
    def merge_into_base(self):
        for layer_idx, expert_id in self.selected_pairs:
            experts = get_sparse_mlp(layer_idx).experts
            gu_key, down_key = self.key_map[(layer_idx, expert_id)]

            experts.gate_up_proj[expert_id].copy_(
                self.params[gu_key].to(experts.gate_up_proj.dtype)
            )
            experts.down_proj[expert_id].copy_(
                self.params[down_key].to(experts.down_proj.dtype)
            )

## 19 — Surgical memory estimate

In [ ]:
if SELECTED_EXPERTS:
    bank_preview = SurgicalExpertBank(SELECTED_EXPERTS)
    n = bank_preview.trainable_parameter_count

    print("Selected expert blocks:", len(SELECTED_EXPERTS))
    print(f"Trainable params: {n:,} ({n/1e6:.2f}M)")
    print(f"FP32 trainable bank: {n*4/2**30:.3f} GiB")
    print(f"Conservative params+grads+Adam states: {n*16/2**30:.3f} GiB")

    del bank_preview
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("No automatic candidates yet. Complete the causal search first.")

## 20 — Optional full-rank selected-expert training

Disabled by default.

Replace `TRAIN_ROWS` with a proper training split that is separate from causal
selection and held-out evaluation.

Recommended first run:

- 4–12 selected expert blocks
- microbatch 1
- sequence <=1024
- gradient accumulation 8–32
- AdamW
- learning rate 5e-6 to 2e-5
- gradient checkpointing enabled

In [ ]:
TRAIN_ROWS = [
    {
        "prompt": "A flex child refuses to shrink and causes horizontal overflow. What CSS declaration should you try?",
        "reference": "min-width: 0;",
    },
    {
        "prompt": "In React, what hook is normally used to hold local state?",
        "reference": "useState",
    },
    {
        "prompt": "What CSS property creates spacing between children in flex and grid layouts?",
        "reference": "gap",
    },
]

RUN_SURGICAL_TRAINING = False
TRAIN_EPOCHS = 1
GRAD_ACCUM_STEPS = 8
LEARNING_RATE = 1e-5
MAX_TRAIN_STEPS = 50
MAX_TRAIN_TOKENS = 1024

In [ ]:
def make_training_case(prompt, reference, max_length=1024):
    prefix_text = chat_prefix_text(prompt)
    prefix_ids = tokenizer.encode(prefix_text, add_special_tokens=False)
    full_ids = tokenizer.encode(prefix_text + " " + reference, add_special_tokens=False)

    if len(full_ids) > max_length:
        raise ValueError(f"{len(full_ids)} tokens > {max_length}")

    start = 0
    for a, b in zip(prefix_ids, full_ids):
        if a != b:
            break
        start += 1

    if start <= 0 or start >= len(full_ids):
        raise ValueError("Could not identify answer boundary.")

    input_ids = torch.tensor(
        full_ids,
        device="cuda:0",
        dtype=torch.long,
    ).unsqueeze(0)

    attention_mask = torch.ones_like(input_ids)

    pred_positions = torch.arange(
        start - 1,
        len(full_ids) - 1,
        device="cuda:0",
        dtype=torch.long,
    )

    targets = torch.tensor(
        full_ids[start:],
        device="cuda:0",
        dtype=torch.long,
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "pred_positions": pred_positions,
        "targets": targets,
    }

TRAIN_CASES = [
    make_training_case(
        r["prompt"],
        r["reference"],
        MAX_TRAIN_TOKENS,
    )
    for r in TRAIN_ROWS
]

print("Prepared training cases:", len(TRAIN_CASES))

## 21 — Run training only when explicitly enabled

In [ ]:
TRAIN_HISTORY = []

if RUN_SURGICAL_TRAINING:
    if not SELECTED_EXPERTS:
        raise RuntimeError("No selected experts.")

    bank = SurgicalExpertBank(SELECTED_EXPERTS)
    bank.install()
    bank.train()

    for p in model.parameters():
        p.requires_grad_(False)

    model.config.use_cache = False
    model.train()

    try:
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )
    except TypeError:
        model.gradient_checkpointing_enable()

    optimizer = torch.optim.AdamW(
        bank.parameters(),
        lr=LEARNING_RATE,
        betas=(0.9, 0.95),
        weight_decay=0.01,
    )

    optimizer.zero_grad(set_to_none=True)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    raw_step = 0
    update_step = 0

    for epoch in range(TRAIN_EPOCHS):
        for case_idx, case in enumerate(TRAIN_CASES):
            raw_step += 1

            with torch.autocast("cuda", dtype=torch.bfloat16):
                out = model(
                    input_ids=case["input_ids"],
                    attention_mask=case["attention_mask"],
                    use_cache=False,
                    logits_to_keep=case["pred_positions"],
                    return_dict=True,
                )

                logits = out.logits.float()
                loss = F.cross_entropy(
                    logits.reshape(-1, logits.shape[-1]),
                    case["targets"].reshape(-1),
                )
                scaled = loss / GRAD_ACCUM_STEPS

            scaled.backward()

            final_available_case = (
                epoch == TRAIN_EPOCHS - 1
                and case_idx == len(TRAIN_CASES) - 1
            )

            if (
                raw_step % GRAD_ACCUM_STEPS == 0
                or raw_step >= MAX_TRAIN_STEPS
                or final_available_case
            ):
                grad_norm = torch.nn.utils.clip_grad_norm_(
                    bank.parameters(),
                    1.0,
                )

                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

                update_step += 1

                TRAIN_HISTORY.append({
                    "step": raw_step,
                    "update_step": update_step,
                    "loss": float(loss.detach().item()),
                    "grad_norm": float(grad_norm),
                })

                print(
                    f"step={raw_step} update={update_step} "
                    f"loss={loss.item():.4f} grad_norm={float(grad_norm):.3f}"
                )

            del out, logits, loss, scaled

            if raw_step >= MAX_TRAIN_STEPS:
                break

        if raw_step >= MAX_TRAIN_STEPS:
            break

    torch.cuda.synchronize()

    print(
        "Training peak allocated:",
        f"{torch.cuda.max_memory_allocated()/2**30:.2f} GiB",
    )
else:
    print("Training skipped.")

## 22 — Post-training target/control check + compact save

In [ ]:
if RUN_SURGICAL_TRAINING:
    model.eval()
    bank.eval()

    AFTER_TRAIN_NLL = score_cached_batch()
    post_delta = AFTER_TRAIN_NLL - BASE_NLL

    print(
        "Target mean ΔNLL after surgery:",
        float(post_delta[target_mask].mean()),
    )
    print(
        "Control mean ΔNLL after surgery:",
        float(post_delta[control_mask].mean()),
    )

    bank_path = RESULTS / "surgical_expert_bank.pt"

    torch.save(
        {
            "model_id": MODEL_ID,
            "selected_experts": SELECTED_EXPERTS,
            "state_dict": {
                k: v.detach().cpu()
                for k, v in bank.state_dict().items()
            },
            "train_history": TRAIN_HISTORY,
            "learning_rate": LEARNING_RATE,
            "grad_accum_steps": GRAD_ACCUM_STEPS,
        },
        bank_path,
    )

    print("Saved:", bank_path)
    print(f"Compact bank size: {bank_path.stat().st_size/2**20:.2f} MiB")

## 23 — Experiment manifest and results archive

In [ ]:
import json, shutil
from datetime import datetime, timezone

free_b, total_b = torch.cuda.mem_get_info()

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "model_path": str(MODEL_PATH),
    "backend": "PyTorch/Transformers BF16",
    "hardware": {
        "gpu": torch.cuda.get_device_name(0),
        "gpu_vram_gib": torch.cuda.get_device_properties(0).total_memory/2**30,
        "logical_cpus": os.cpu_count(),
        "ram_gib": psutil.virtual_memory().total/2**30,
    },
    "software": {
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "cuda": torch.version.cuda,
    },
    "architecture": {
        "hidden_size": int(cfg.hidden_size),
        "layers": int(cfg.num_hidden_layers),
        "sparse_layers": list(map(int, SPARSE_LAYERS)),
        "experts": int(cfg.num_experts),
        "top_k": int(cfg.num_experts_per_tok),
        "expert_width": int(cfg.moe_intermediate_size),
        "params_per_expert": int(params_per_expert),
    },
    "search": {
        "control_penalty": CONTROL_PENALTY,
        "top_layers": TOP_LAYERS,
        "search_seeds": SEARCH_SEEDS,
        "initial_group_size": INITIAL_GROUP_SIZE,
        "beam_width": BEAM_WIDTH,
        "eval_examples": len(eval_df),
    },
    "selected_experts": SELECTED_EXPERTS,
    "training_enabled": RUN_SURGICAL_TRAINING,
    "gpu_end": {
        "allocated_gib": torch.cuda.memory_allocated()/2**30,
        "reserved_gib": torch.cuda.memory_reserved()/2**30,
        "free_gib": free_b/2**30,
        "peak_gib": torch.cuda.max_memory_allocated()/2**30,
    },
}

(RESULTS / "manifest.json").write_text(json.dumps(manifest, indent=2))

archive = shutil.make_archive(
    str(RESULTS),
    "zip",
    root_dir=RESULTS,
)

print("Results:", RESULTS)
print("Archive:", archive)

display(
    final_robust[
        [
            "layer","expert",
            "target_delta_nll",
            "control_delta_nll",
            "causal_specificity",
            "renorm_causal_specificity",
            "ci_2.5","ci_97.5",
            "p_positive",
        ]
    ].head(20)
)

# Publication-quality protocol

Before claiming a result:

1. replace the toy probe set with a proper selection dataset;
2. use a disjoint held-out target/control evaluation set;
3. repeat expert search with multiple randomized group orderings;
4. compare matched trainable-parameter/data budgets:
   - random experts,
   - most-routed experts,
   - gradient/saliency,
   - task/success association,
   - causal expert surgery;
5. report target gain, general retention, forgetting, parameter count,
   optimizer memory, search compute, and routing-vs-causal rank correlation;
6. test both intervention semantics and a small number of coalitions.

The falsifiable question is:

> At the same parameter and data budget, does measured causal contribution
> select a better sparse adaptation circuit than observational expert-selection
> methods?